# Real-time Driver Monitoring System using AI

It is a real-time computer vision-based driver monitoring system that detects drowsiness, distraction, and unsafe behavior using MediaPipe and YOLO, and generates alerts to prevent accidents.

In [6]:
import cv2
import mediapipe as mp
import numpy as np
import time
from ultralytics import YOLO
import pygame
import os
import csv

In [7]:
pygame.mixer.init()
current_sound = None

def play_sound(file):
    global current_sound
    if not os.path.exists(file):
        return
    if current_sound != file:
        pygame.mixer.music.load(file)
        pygame.mixer.music.play()
        current_sound = file

def stop_sound():
    global current_sound
    pygame.mixer.music.stop()
    current_sound = None

In [8]:
log_file = "driver_log.csv"

if not os.path.exists(log_file):
    with open(log_file, "w", newline="") as f:
        csv.writer(f).writerow(["Time", "Event", "Risk"])

def log_event(event, risk):
    with open(log_file, "a", newline="") as f:
        csv.writer(f).writerow([time.strftime("%H:%M:%S"), event, risk])

In [9]:
model = YOLO("yolov8n.pt")

mp_face = mp.solutions.face_mesh
face_mesh = mp_face.FaceMesh(refine_landmarks=True)

In [10]:
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

UPPER_LIP = 13
LOWER_LIP = 14
LEFT_MOUTH = 78
RIGHT_MOUTH = 308

In [11]:
def calculate_ear(landmarks, eye_idx, w, h):
    pts = [(int(landmarks[i].x * w), int(landmarks[i].y * h)) for i in eye_idx]
    p1, p2, p3, p4, p5, p6 = pts

    v1 = np.linalg.norm(np.array(p2) - np.array(p6))
    v2 = np.linalg.norm(np.array(p3) - np.array(p5))
    h1 = np.linalg.norm(np.array(p1) - np.array(p4))

    return (v1 + v2) / (2.0 * h1)


def calculate_mar(landmarks, w, h):
    upper = np.array([landmarks[UPPER_LIP].x * w, landmarks[UPPER_LIP].y * h])
    lower = np.array([landmarks[LOWER_LIP].x * w, landmarks[LOWER_LIP].y * h])
    left = np.array([landmarks[LEFT_MOUTH].x * w, landmarks[LEFT_MOUTH].y * h])
    right = np.array([landmarks[RIGHT_MOUTH].x * w, landmarks[RIGHT_MOUTH].y * h])

    return np.linalg.norm(upper - lower) / np.linalg.norm(left - right)

In [12]:
EAR_THRESHOLD = 0.25
MAR_THRESHOLD = 0.6
DROWSY_TIME = 1.2
LOOK_THRESHOLD = 0.035

blink_start = None
look_start = None

last_alert_time = 0
cooldown = 3

In [13]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print(" Camera not found")

In [14]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    h, w, _ = frame.shape

    phone = False
    drowsy = False
    yawn = False
    distracted = False

    # YOLO
    results = model(frame, verbose=False)

    for r in results:
        for box in r.boxes:
            cls = int(box.cls[0])
            name = model.names[cls]

            x1, y1, x2, y2 = map(int, box.xyxy[0])

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
            cv2.putText(frame, name, (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

            if name == "cell phone":
                phone = True

    # FACE
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    res = face_mesh.process(rgb)

    if res.multi_face_landmarks:
        for face in res.multi_face_landmarks:

            # EAR
            ear = (calculate_ear(face.landmark, LEFT_EYE, w, h) +
                   calculate_ear(face.landmark, RIGHT_EYE, w, h)) / 2

            # MAR
            mar = calculate_mar(face.landmark, w, h)

            # DROWSY
            if ear < EAR_THRESHOLD:
                if blink_start is None:
                    blink_start = time.time()
                elif time.time() - blink_start > DROWSY_TIME:
                    drowsy = True
            else:
                blink_start = None

            # YAWN
            if mar > MAR_THRESHOLD:
                yawn = True

            # DISTRACTION
            left_eye_x = np.mean([face.landmark[i].x for i in LEFT_EYE])
            right_eye_x = np.mean([face.landmark[i].x for i in RIGHT_EYE])

            eye_center_x = (left_eye_x + right_eye_x) / 2
            face_center_x = face.landmark[1].x

            diff = abs(eye_center_x - face_center_x)

            if diff > LOOK_THRESHOLD:
                if look_start is None:
                    look_start = time.time()
                elif time.time() - look_start > 1.2:
                    distracted = True
            else:
                look_start = None

            cv2.putText(frame, f"LookDiff: {diff:.3f}", (30, 140),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,0,0), 2)

    # RISK
    risk = 0
    if drowsy: risk += 3
    if phone: risk += 2
    if distracted: risk += 2
    if yawn: risk += 1

    if risk >= 5:
        level = "HIGH"
    elif risk >= 3:
        level = "MEDIUM"
    else:
        level = "LOW"

    # ALERTS
    current_time = time.time()

    if current_time - last_alert_time > cooldown:
        if drowsy:
            play_sound("drowsy.wav")
            log_event("Drowsiness", risk)
            last_alert_time = current_time

        elif phone:
            play_sound("phone.wav")
            log_event("Phone Usage", risk)
            last_alert_time = current_time

        elif distracted:
            play_sound("attention.wav")
            log_event("Distracted", risk)
            last_alert_time = current_time

        elif yawn:
            play_sound("attention.wav")
            log_event("Yawning", risk)
            last_alert_time = current_time

        else:
            stop_sound()

    # DISPLAY
    cv2.putText(frame, f"RISK: {level}", (30, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

    cv2.imshow("Driver Monitoring System", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()